# YouTube Shorts Bulk Analysis - Advanced
## Find parent long-form videos using ytInitialData extraction
1. Extract shorts from dataset
2. Fetch YouTube HTML and extract ytInitialData JSON
3. Find watchEndpoint source video IDs
4. Cross-reference with description links for verification

In [ ]:
# ⚙️  CONFIGURATION - TOGGLE HERE
USE_SAMPLE = True  # Set to True to test with sample, False to process all shorts
SAMPLE_SIZE = 5 # Number of shorts to process if USE_SAMPLE=True

# 📅 DATE FILTER - Only analyze shorts from this year onwards
START_YEAR = 2021  # Only process shorts from 2025 onwards
END_YEAR = 2026    # Process up to this year (2026)

GCP_PROJECT_ID   = "gen-lang-client-0792749758"
GCP_LOCATION     = "us-central1"
GCP_BUCKET       = "afb_showreel"

# ⚡ PERFORMANCE / POLITENESS KNOBS  (tune speed vs. YouTube rate-limit risk)
# Effective request rate ≈ CONCURRENCY / avg(MIN_DELAY, MAX_DELAY).
# Defaults below give ~4 tabs with a 0.5-1.5s jittered pause => ~3-4 req/s,
# which is brisk but stays under YouTube's bot-detection threshold.
CONCURRENCY          = 4      # parallel browser tabs. 3-6 is the safe sweet spot.
MIN_DELAY            = 0.5    # min polite pause per request (seconds)
MAX_DELAY            = 1.5    # max polite pause per request (seconds)
SELECTOR_TIMEOUT_MS  = 8000   # how long to wait for the overlay link (was 15000)
NAV_TIMEOUT_MS       = 25000  # page navigation timeout
BLOCK_RESOURCES      = True   # abort image/media/font loads -> much faster DOM
ENABLE_YTDLP_FALLBACK = True  # description-link fallback when overlay is absent
YTDLP_CONCURRENCY    = 3      # parallel yt-dlp subprocesses (separate budget)
YTDLP_TIMEOUT_S      = 20     # per-video yt-dlp timeout (was 30)
CHECKPOINT_EVERY     = 250    # save partial results to disk every N shorts

print(f"Configuration:")
print(f"  USE_SAMPLE: {USE_SAMPLE}")
print(f"  SAMPLE_SIZE: {SAMPLE_SIZE}")
print(f"  DATE RANGE: {START_YEAR}-01-01 to {END_YEAR}-12-31")
print(f"  CONCURRENCY: {CONCURRENCY} tabs | delay {MIN_DELAY}-{MAX_DELAY}s | "
      f"yt-dlp fallback: {ENABLE_YTDLP_FALLBACK}")
print(f"\nTo change:")
print(f"  - Set USE_SAMPLE=False for full dataset analysis")
print(f"  - Adjust START_YEAR/END_YEAR to change date range")
print(f"  - Raise CONCURRENCY / lower delays to go faster (watch for rate limits)")

In [48]:
!pip install -q playwright
# This command installs the browser AND the missing system dependencies (libatk, etc.)
!playwright install --with-deps chromium

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://packages.cloud.google.com/apt gcsfuse-jammy InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: https://packages.cloud.google.com/apt/dists/gcsfuse-jammy/InRelease: Key is stored in legacy trusted.gpg

In [49]:
import pandas as pd
import numpy as np
import json
import random
import re
import subprocess
import requests
from pathlib import Path
from datetime import datetime
import time

# Playwright for rendering the Shorts overlay (JS-hydrated DOM)
from playwright.sync_api import sync_playwright

print("Libraries loaded successfully")
print(f"Started at: {datetime.now()}")

Libraries loaded successfully
Started at: 2026-05-31 17:39:28.344221


In [50]:
# prompt: connect to the bucket and download youtube's metadata videos_metadata.csv

from google.cloud import storage

client = storage.Client(project=GCP_PROJECT_ID)
bucket = client.get_bucket(GCP_BUCKET)
blob = bucket.blob("videos_metadata.csv")
blob.download_to_filename("videos_metadata.csv")
print("Downloaded youtube's metadata videos_metadata.csv")
display(pd.read_csv("videos_metadata.csv").head())
yt_df = pd.read_csv("videos_metadata.csv")


Downloaded youtube's metadata videos_metadata.csv


/tmp/ipykernel_1794/3650638829.py:10: DtypeWarning: Columns (35,62,65) have mixed types. Specify dtype option on import or set low_memory=False.
  display(pd.read_csv("videos_metadata.csv").head())


,videoId,publishedAt,channelId,channelTitle,title,description,tags,categoryId,liveBroadcastContent,defaultLanguage,...,topicCategories,recordingDate,actualStartTime,actualEndTime,scheduledStartTime,scheduledEndTime,concurrentViewers,activeLiveChatId,contentRating_ytRating,is_short
0,eNU5FcmEwTc,2026-02-27T14:16:42Z,UC4TFt-3rlYfhc_WtlWWY30Q,Camihawke,ChatGPT non ha aiutato...,NaN,NaN,22,none,it,...,https://en.wikipedia.org/wiki/Entertainment,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
1,tNbsrpKaMoE,2026-02-26T13:18:36Z,UC4TFt-3rlYfhc_WtlWWY30Q,Camihawke,TIER LIST - Sanremo Edition con @willwoosh,"Il video è ironico, per le informazioni sui ca...",NaN,22,none,it,...,https://en.wikipedia.org/wiki/Entertainment,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,80cTxK_fA_Q,2026-02-25T12:00:28Z,UC4TFt-3rlYfhc_WtlWWY30Q,Camihawke,Dirò una cosa forte!!!,NaN,NaN,22,none,it,...,https://en.wikipedia.org/wiki/Lifestyle_(socio...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
3,AS6paOwhdR0,2026-02-23T12:00:28Z,UC4TFt-3rlYfhc_WtlWWY30Q,Camihawke,Voi cose pensate quando qualcuno non vi rispon...,NaN,NaN,22,none,it,...,https://en.wikipedia.org/wiki/Lifestyle_(socio...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
4,wPFcvFEPSSQ,2026-02-20T12:01:39Z,UC4TFt-3rlYfhc_WtlWWY30Q,Camihawke,"Mai avere paura delle risposte, mai più!",NaN,NaN,22,none,it,...,https://en.wikipedia.org/wiki/Lifestyle_(socio...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True


/tmp/ipykernel_1794/3650638829.py:11: DtypeWarning: Columns (35,62,65) have mixed types. Specify dtype option on import or set low_memory=False.
  yt_df = pd.read_csv("videos_metadata.csv")


In [51]:
import pandas as pd
from pathlib import Path

# Use the already loaded yt_df
if 'yt_df' in globals():
    df = yt_df.copy()
    print(f"✅ Dataset loaded from memory (yt_df)!")
    print(f"Total videos: {len(df)}")

    # Parse publishedAt as datetime and extract year
    if 'publishedAt' in df.columns:
        df['publishedAt'] = pd.to_datetime(df['publishedAt'], errors='coerce')
        df['year'] = df['publishedAt'].dt.year
        print(f"Date range in dataset: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

    # Filter shorts
    if 'is_short' in df.columns:
        all_shorts = df[df['is_short'] == True].copy()
        all_longs = df[df['is_short'] == False].copy()

        print(f"\nTotal shorts: {len(all_shorts)}")
        print(f"Total longs: {len(all_longs)}")

        # Apply YEAR FILTER
        print(f"\n📅 Applying date filter: {START_YEAR} onwards")
        shorts_df = all_shorts[
            (all_shorts['year'] >= START_YEAR) &
            (all_shorts['year'] <= END_YEAR)
        ].copy().reset_index(drop=True)

        print(f"Shorts after date filter: {len(shorts_df)} (from {len(all_shorts)} total)")

        if len(shorts_df) > 0:
            print(f"\nDate range of filtered shorts: {shorts_df['publishedAt'].min()} to {shorts_df['publishedAt'].max()}")
            print(f"\n📊 Year distribution:")
            year_counts = shorts_df['year'].value_counts().sort_index()
            for year, count in year_counts.items():
                print(f"  {int(year)}: {count} shorts")
        else:
            print(f"⚠️  No shorts found in date range {START_YEAR}-{END_YEAR}")

        # Apply STRATIFIED RANDOM SAMPLE by channel title
        if USE_SAMPLE and len(shorts_df) > SAMPLE_SIZE:
            print(f"\n⚠️  USING STRATIFIED RANDOM SAMPLE MODE (by channel)")
            print(f"Sampling {SAMPLE_SIZE} random shorts stratified by channel...\n")

            sampled_list = []
            total_shorts_count = len(shorts_df)
            channels = shorts_df['channelTitle'].unique()

            for channel in channels:
                channel_data = shorts_df[shorts_df['channelTitle'] == channel]
                proportion = len(channel_data) / total_shorts_count
                n_samples = max(1, int(SAMPLE_SIZE * proportion))

                if len(channel_data) > 0:
                    sampled = channel_data.sample(n=min(n_samples, len(channel_data)), random_state=42)
                    sampled_list.append(sampled)

            shorts_df = pd.concat(sampled_list, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

            print(f"✅ Stratified random sample created: {len(shorts_df)} shorts")
            print(f"\n📊 Channel representation in sample:")
            display(shorts_df['channelTitle'].value_counts().head(10))
    else:
        print("❌ No 'is_short' column")
        shorts_df = None
else:
    print(f"❌ yt_df not found in memory. Please run the cell that downloads data from GCS.")
    shorts_df = None

✅ Dataset loaded from memory (yt_df)!
Total videos: 12839
Date range in dataset: 2008-02-14 17:04:02+00:00 to 2026-03-01 13:47:43+00:00

Total shorts: 4356
Total longs: 8483

📅 Applying date filter: 2021 onwards
Shorts after date filter: 4350 (from 4356 total)

Date range of filtered shorts: 2021-06-04 06:57:13+00:00 to 2026-03-01 13:47:43+00:00

📊 Year distribution:
  2021: 270 shorts
  2022: 852 shorts
  2023: 919 shorts
  2024: 1008 shorts
  2025: 1056 shorts
  2026: 245 shorts

⚠️  USING STRATIFIED RANDOM SAMPLE MODE (by channel)
Sampling 5 random shorts stratified by channel...

✅ Stratified random sample created: 25 shorts

📊 Channel representation in sample:


,count
channelTitle,
THOMAS BASILICO,1
Sara La Rusca,1
Camihawke,1
AlicelikeAudrey,1
Il Matricomio,1
La casa di Mattia,1
Christian Manzoni,1
Gli Autogol,1
Bella Gianda Official,1


In [52]:
short_df = shorts_df[shorts_df['is_short'] == True].copy()

In [53]:
# =========================================================================
# PLAYWRIGHT-BASED EXTRACTION
# Renders the Short's JS overlay and reads the
# <a class="ytReelMultiFormatLinkViewModelEndpoint"> element directly.
# Records the connection whether it points to a /watch?v= (long) or
# a /shorts/ (short) video.
# =========================================================================

# Consent cookie to bypass YouTube's EU/Italy consent interstitial
# (consent.youtube.com). Without this the overlay never renders.
YT_CONSENT_COOKIE = {
    "name": "SOCS",
    "value": "CAISEwgDEgk0ODE3Nzk3MjQaAmVuIAEaBgiA_LyaBg",
    "domain": ".youtube.com",
    "path": "/",
}

MULTIFORMAT_SELECTOR = "a.ytReelMultiFormatLinkViewModelEndpoint"


def start_browser(playwright, headless=True):
    """Launch a Chromium browser + context with consent cookie pre-set."""
    browser = playwright.chromium.launch(headless=headless)
    context = browser.new_context(
        user_agent=("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36"),
        locale="en-US",
    )
    context.add_cookies([YT_CONSENT_COOKIE])
    return browser, context


def parse_link_target(href):
    """
    Given an overlay href, return (link_type, linked_video_id).
      link_type: 'long'  for /watch?v=...
                 'short' for /shorts/...
                 None    for anything else
    """
    if not href:
        return None, None
    m = re.search(r"/watch\?v=([a-zA-Z0-9_-]{11})", href)
    if m:
        return "long", m.group(1)
    m = re.search(r"/shorts/([a-zA-Z0-9_-]{11})", href)
    if m:
        return "short", m.group(1)
    return None, None


def get_multiformat_link(page, short_id, timeout_ms=15000):
    """
    Navigate to a Short and extract the multi-format link overlay element.

    Records the connection regardless of whether it links to a long video
    (/watch?v=) or another short (/shorts/).

    Returns a dict:
      {
        'href':  '/watch?v=...' or '/shorts/...' or None,
        'label': aria-label / title text,
        'link_type': 'long' | 'short' | None,
        'linked_video_id': the 11-char id of the linked video (long or short),
      }
    or None if the element never rendered.
    """
    url = f"https://www.youtube.com/shorts/{short_id}"
    try:
        page.goto(url, wait_until="domcontentloaded", timeout=30000)

        if "consent.youtube.com" in page.url:
            return None  # consent bypass failed

        page.wait_for_selector(MULTIFORMAT_SELECTOR, state="attached", timeout=timeout_ms)

        rows = page.eval_on_selector_all(
            MULTIFORMAT_SELECTOR,
            "els => els.map(e => ({href: e.getAttribute('href'), "
            "label: e.getAttribute('aria-label')}))"
        )

        # Prefer a /watch?v= (long video) link; otherwise the first link to a
        # *different* video than the short itself; otherwise the self-link.
        chosen = None
        for r in rows:
            lt, vid = parse_link_target(r.get("href"))
            if lt == "long":
                chosen = r
                break
        if chosen is None:
            for r in rows:
                lt, vid = parse_link_target(r.get("href"))
                if vid and vid != short_id:
                    chosen = r
                    break
        if chosen is None and rows:
            chosen = rows[0]
        if chosen is None:
            return None

        href = chosen.get("href")
        link_type, linked_video_id = parse_link_target(href)

        return {
            "href": href,
            "label": chosen.get("label"),
            "link_type": link_type,
            "linked_video_id": linked_video_id,
        }
    except Exception:
        return None


# Fallback: download JSON metadata via yt-dlp (for description links)
def download_video_json(video_id):
    """Download complete JSON metadata for a video using yt-dlp"""
    try:
        cmd = ['yt-dlp', '--dump-json', '--skip-download',
               f'https://www.youtube.com/watch?v={video_id}']
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode == 0:
            return json.loads(result.stdout)
    except Exception:
        pass
    return None


def extract_youtube_links_from_description(text, exclude_id=None):
    """Extract YouTube video IDs from description text (fallback method)"""
    if not text:
        return []
    links = []
    patterns = [
        r'(?:https?://)?(?:www\.)?youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})',
        r'(?:https?://)?youtu\.be/([a-zA-Z0-9_-]{11})',
        r'(?:https?://)?(?:www\.)?youtube\.com/shorts/([a-zA-Z0-9_-]{11})'
    ]
    for pattern in patterns:
        for match in re.finditer(pattern, text):
            vid_id = match.group(1)
            if exclude_id is None or vid_id != exclude_id:
                links.append(vid_id)
    return list(dict.fromkeys(links))


print("✅ Helper functions defined - Playwright overlay extractor (long + short) + yt-dlp fallback")

✅ Helper functions defined - Playwright overlay extractor (long + short) + yt-dlp fallback


# Function Definition

In [ ]:
# =========================================================================
# CONCURRENT, RATE-LIMITED OVERLAY EXTRACTION  (Colab Enterprise / Linux)
# -------------------------------------------------------------------------
# Same logic as before (render the Short's JS overlay, read the
# <a class="ytReelMultiFormatLinkViewModelEndpoint"> link, fall back to the
# description via yt-dlp), but now run by a small pool of worker tabs.
#
# Speed-ups vs. the old one-at-a-time loop:
#   * CONCURRENCY tabs pull from a shared queue (was: 1 page, fully serial).
#   * Image/media/font requests are aborted -> the DOM hydrates much faster.
#   * yt-dlp fallback runs as a NON-blocking async subprocess with its own
#     small concurrency budget (was: a blocking 30s subprocess.run per short).
#   * Selector/nav timeouts trimmed.
# Politeness vs. rate limits:
#   * A jittered MIN_DELAY..MAX_DELAY pause after every request per worker, so
#     the aggregate rate stays ~CONCURRENCY / avg_delay req/s. Tune in cell 1.
#   * Periodic checkpoint to ./shorts_analysis/_checkpoint.parquet so a long
#     full-dataset run on ephemeral Colab storage is never fully lost.
# =========================================================================
import asyncio
import re
import json
import random
from pathlib import Path

import pandas as pd
from playwright.async_api import async_playwright

YT_CONSENT_COOKIE = {
    "name": "SOCS",
    "value": "CAISEwgDEgk0ODE3Nzk3MjQaAmVuIAEaBgiA_LyaBg",
    "domain": ".youtube.com",
    "path": "/",
}
MULTIFORMAT_SELECTOR = "a.ytReelMultiFormatLinkViewModelEndpoint"

# Resource types to drop so pages render faster (DOM attrs are all we need).
_BLOCKED_RESOURCE_TYPES = {"image", "media", "font"}


def parse_link_target(href):
    if not href:
        return None, None
    m = re.search(r"/watch\?v=([a-zA-Z0-9_-]{11})", href)
    if m:
        return "long", m.group(1)
    m = re.search(r"/shorts/([a-zA-Z0-9_-]{11})", href)
    if m:
        return "short", m.group(1)
    return None, None


def extract_youtube_links_from_description(text, exclude_id=None):
    if not text:
        return []
    links = []
    patterns = [
        r'(?:https?://)?(?:www\.)?youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})',
        r'(?:https?://)?youtu\.be/([a-zA-Z0-9_-]{11})',
        r'(?:https?://)?(?:www\.)?youtube\.com/shorts/([a-zA-Z0-9_-]{11})'
    ]
    for pattern in patterns:
        for match in re.finditer(pattern, text):
            vid_id = match.group(1)
            if exclude_id is None or vid_id != exclude_id:
                links.append(vid_id)
    return list(dict.fromkeys(links))


async def download_video_json_async(video_id, sem):
    """Non-blocking yt-dlp metadata fetch, bounded by `sem`."""
    async with sem:
        proc = None
        try:
            proc = await asyncio.create_subprocess_exec(
                'yt-dlp', '--dump-json', '--skip-download', '--no-warnings',
                f'https://www.youtube.com/watch?v={video_id}',
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.DEVNULL,
            )
            out, _ = await asyncio.wait_for(proc.communicate(), timeout=YTDLP_TIMEOUT_S)
            if proc.returncode == 0 and out:
                return json.loads(out.decode('utf-8', 'ignore'))
        except Exception:
            if proc is not None:
                try:
                    proc.kill()
                except Exception:
                    pass
        return None


async def get_multiformat_link(page, short_id):
    url = f"https://www.youtube.com/shorts/{short_id}"
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=NAV_TIMEOUT_MS)
        if "consent.youtube.com" in page.url:
            return None
        await page.wait_for_selector(MULTIFORMAT_SELECTOR, state="attached",
                                     timeout=SELECTOR_TIMEOUT_MS)
        rows = await page.eval_on_selector_all(
            MULTIFORMAT_SELECTOR,
            "els => els.map(e => ({href: e.getAttribute('href'), "
            "label: e.getAttribute('aria-label')}))"
        )
        chosen = next((r for r in rows if parse_link_target(r.get("href"))[0] == "long"), None)
        if not chosen:
            chosen = next((r for r in rows
                           if parse_link_target(r.get("href"))[1] not in (None, short_id)), None)
        if not chosen and rows:
            chosen = rows[0]
        if not chosen:
            return None
        href = chosen.get("href")
        link_type, linked_video_id = parse_link_target(href)
        return {
            "href": href,
            "label": chosen.get("label"),
            "link_type": link_type,
            "linked_video_id": linked_video_id,
        }
    except Exception:
        return None


async def _route_filter(route):
    if route.request.resource_type in _BLOCKED_RESOURCE_TYPES:
        await route.abort()
    else:
        await route.continue_()


async def _worker(name, queue, context, results, state, total, ytdlp_sem):
    page = await context.new_page()
    if BLOCK_RESOURCES:
        await page.route("**/*", _route_filter)
    ckpt_path = Path('./shorts_analysis/_checkpoint.parquet')
    try:
        while True:
            try:
                orig_idx, video_id = queue.get_nowait()
            except asyncio.QueueEmpty:
                break

            link_info = await get_multiformat_link(page, video_id)

            res = {
                'orig_idx': orig_idx,
                'short_id': video_id,
                'status': 'no_connection',
                'connection_type': 'none',
                'connected_id': None,
                'overlay_label': 'N/A',
                'detection_method': None,
            }

            if (link_info and link_info.get('linked_video_id')
                    and link_info['linked_video_id'] != video_id):
                res.update({
                    'status': 'connected',
                    'connection_type': link_info['link_type'],
                    'connected_id': link_info['linked_video_id'],
                    'overlay_label': link_info.get('label') or 'N/A',
                    'detection_method': 'overlay',
                })

            if res['status'] == 'no_connection' and ENABLE_YTDLP_FALLBACK:
                vj = await download_video_json_async(video_id, ytdlp_sem)
                if vj:
                    fb = extract_youtube_links_from_description(vj.get('description', ''), video_id)
                    if fb:
                        res.update({
                            'status': 'connected',
                            'connection_type': 'long',
                            'connected_id': fb[0],
                            'overlay_label': 'From Description',
                            'detection_method': 'description',
                        })

            results.append(res)
            state['done'] += 1
            done = state['done']
            if done % 25 == 0 or done == total:
                print(f"  [{done}/{total}] processed ({len(results)} records)", flush=True)
            if done % CHECKPOINT_EVERY == 0:
                try:
                    pd.DataFrame(results).to_parquet(ckpt_path, index=False)
                except Exception:
                    pass

            # politeness jitter -> keeps aggregate rate under YouTube's radar
            await asyncio.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
    finally:
        await page.close()


async def process_shorts_async(shorts_df):
    total = len(shorts_df)
    Path('./shorts_analysis').mkdir(parents=True, exist_ok=True)
    print(f"Processing {total} shorts with {CONCURRENCY} concurrent tabs "
          f"(delay {MIN_DELAY}-{MAX_DELAY}s, yt-dlp fallback={ENABLE_YTDLP_FALLBACK})...")

    queue = asyncio.Queue()
    for orig_idx, short in shorts_df.reset_index(drop=True).iterrows():
        queue.put_nowait((orig_idx, str(short['videoId'])))

    results = []
    state = {'done': 0}
    ytdlp_sem = asyncio.Semaphore(YTDLP_CONCURRENCY)

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                        "(KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36"),
            locale="en-US",
        )
        await context.add_cookies([YT_CONSENT_COOKIE])

        workers = [
            asyncio.create_task(
                _worker(f"w{i}", queue, context, results, state, total, ytdlp_sem)
            )
            for i in range(CONCURRENCY)
        ]
        await asyncio.gather(*workers)
        await browser.close()

    # restore original ordering, then drop the helper column
    out = pd.DataFrame(results).sort_values('orig_idx').drop(columns=['orig_idx']).reset_index(drop=True)
    return out


if 'shorts_df' in globals() and shorts_df is not None:
    import time as _t
    _start = _t.time()
    results_df = await process_shorts_async(shorts_df)
    print(f"\n✅ Done in {_t.time() - _start:.1f}s "
          f"({len(results_df)} shorts, "
          f"{int((results_df['status'] == 'connected').sum())} connected)")
    display(results_df.head())
else:
    print("Error: 'shorts_df' not found.")

In [55]:
# Display the connection dataset
print("\n" + "="*80)
print("SHORT → VIDEO CONNECTION DATASET")
print("="*80)

if results_df is not None and len(results_df) > 0:
    # Define the columns we want to display
    target_cols = ['short_id', 'connection_type', 'connected_id', 'overlay_label', 'detection_method']

    # Ensure all target columns exist to avoid KeyError, fill missing with 'N/A'
    for col in target_cols:
        if col not in results_df.columns:
            results_df[col] = "N/A"

    # ---- 1) The captured overlay element + recorded connection for EVERY short ----
    conn_dataset = results_df[target_cols].copy()
    conn_dataset.columns = ['Short ID', 'Type', 'Connected ID', 'Overlay label', 'Method']

    print(f"\n🔗 CAPTURED CONNECTIONS (all {len(conn_dataset)} shorts):\n")
    with pd.option_context('display.max_colwidth', 40, 'display.width', 200):
        print(conn_dataset.to_string(index=False))

    # ---- 2) Breakdown by connection type ----
    print("\n" + "-"*80)
    print("\n📊 CONNECTION TYPE BREAKDOWN:")
    type_counts = results_df['connection_type'].value_counts()
    for t, c in type_counts.items():
        print(f"  {t:>6}: {c} ({c/len(results_df)*100:.1f}%)")

    # ---- 3) All recorded connections with clickable links ----
    connected = results_df[results_df['status'] == 'connected'].copy()
    print("\n" + "-"*80)
    if len(connected) > 0:
        print(f"\n✅ {len(connected)} SHORTS WITH A RECORDED CONNECTION\n")
        for idx, (_, row) in enumerate(connected.iterrows(), 1):
            emoji = "📹" if row['connection_type'] == 'long' else "📱"
            short_url = f"https://www.youtube.com/shorts/{row['short_id']}"
            conn_url = f"https://www.youtube.com/watch?v={row['connected_id']}" if row['connection_type'] == 'long' else f"https://www.youtube.com/shorts/{row['connected_id']}"

            print(f"{idx}. SHORT : {row['short_id']}")
            print(f"   📱 {short_url}")
            print(f"   ↓ {str(row['connection_type']).upper()} link ({row['detection_method']})")
            print(f"   {emoji} {row['connected_id']}")
            print(f"   🔗 {conn_url}")
            print()
    else:
        print("\n❌ No connections recorded in this sample.")
else:
    print("\n❌ No results. Run the processing cell first.")


SHORT → VIDEO CONNECTION DATASET

🔗 CAPTURED CONNECTIONS (all 25 shorts):

   Short ID Type Connected ID Overlay label  Method
bkZeNyvXPe0 none          NaN           N/A     NaN
lf6TQsniXAM none          NaN           N/A     NaN
eNU5FcmEwTc none          NaN           N/A     NaN
Df-9SmBp8NQ none          NaN           N/A     NaN
R-Zn3xAAYqw none          NaN           N/A     NaN
HzH6s2_Cpeg none          NaN           N/A     NaN
b4t6fdYh8ak long  gE2Phwx-nKg           N/A overlay
gs7oJh6kJcY none          NaN           N/A     NaN
F9bR34COjEY none          NaN           N/A     NaN
b8SjcaneGrw none          NaN           N/A     NaN
VEBxxBohLHo long  WqRxEf7okik           N/A overlay
k3QaVA0UleY none          NaN           N/A     NaN
JbJXD-EMs-g none          NaN           N/A     NaN
GRgp3BScD44 none          NaN           N/A     NaN
3mA4U7FNS80 none          NaN           N/A     NaN
62_wzchaX98 none          NaN           N/A     NaN
Y5j4lazJzzo none          NaN           

In [56]:
import pandas as pd
import json
from pathlib import Path

# Define output paths
output_dir = Path('./shorts_analysis')
output_dir.mkdir(parents=True, exist_ok=True)
pq_metadata_path = output_dir / 'shorts_metadata_aggregated.parquet'

print("\n" + "="*80)
print("METADATA AGGREGATION & PARQUET EXPORT")
print("="*80)

# If we have results_df from the previous cell, we can use that data
if 'results_df' in globals() and results_df is not None:
    print(f"\n✅ Found results_df with {len(results_df)} records.")

    # In a real scenario, you might want to merge this with deeper JSON metadata
    # For now, we will save the current results as the primary metadata source
    full_json_df = results_df.copy()

    # Save to Parquet
    full_json_df.to_parquet(pq_metadata_path, index=False)

    print(f"✅ Dataframe saved to Parquet: {pq_metadata_path}")
    display(full_json_df.head())
else:
    print("\u26a0 No results_df found. Please run the processing cell first.")


METADATA AGGREGATION & PARQUET EXPORT

✅ Found results_df with 25 records.
✅ Dataframe saved to Parquet: shorts_analysis/shorts_metadata_aggregated.parquet


,short_id,status,connection_type,connected_id,detection_method,overlay_label
0,bkZeNyvXPe0,no_connection,none,NaN,NaN,N/A
1,lf6TQsniXAM,no_connection,none,NaN,NaN,N/A
2,eNU5FcmEwTc,no_connection,none,NaN,NaN,N/A
3,Df-9SmBp8NQ,no_connection,none,NaN,NaN,N/A
4,R-Zn3xAAYqw,no_connection,none,NaN,NaN,N/A


In [57]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import json

if results_df is not None:
    output_dir = Path('./shorts_analysis')
    output_dir.mkdir(exist_ok=True)

    # Prepare filename suffix
    date_suffix = f'_{START_YEAR}-{END_YEAR}'
    mode_suffix = '_sample' if USE_SAMPLE else '_full'
    suffix = date_suffix + mode_suffix

    # 1. Export Full Results as CSV
    csv_path = output_dir / f'shorts_connections{suffix}.csv'
    results_df.to_csv(csv_path, index=False)
    print(f'\n✅ Full results exported to CSV: {csv_path}')

    # 2. Export Connection Mapping as CSV
    connected_df = results_df[results_df['status'] == 'connected']
    if len(connected_df) > 0:
        mapping_csv_path = output_dir / f'shorts_connection_mapping{suffix}.csv'
        connected_df.to_csv(mapping_csv_path, index=False)
        print(f'✅ Connection mappings exported to CSV: {mapping_csv_path}')

    # 3. Export Summary Stats (JSON)
    stats_path = output_dir / f'summary_stats{suffix}.json'
    type_counts = results_df['connection_type'].value_counts().to_dict()
    stats = {
        'date_range': f'{START_YEAR}-{END_YEAR}',
        'mode': 'sample' if USE_SAMPLE else 'full',
        'total_shorts': len(results_df),
        'shorts_connected': len(connected_df),
        'percentage_connected': round(len(connected_df)/len(results_df)*100, 2) if len(results_df) > 0 else 0,
        'connection_type_counts': {str(k): int(v) for k, v in type_counts.items()},
        'analysis_date': datetime.now().isoformat()
    }

    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f'✅ Stats exported to: {stats_path}')


✅ Full results exported to CSV: shorts_analysis/shorts_connections_2021-2026_sample.csv
✅ Connection mappings exported to CSV: shorts_analysis/shorts_connection_mapping_2021-2026_sample.csv
✅ Stats exported to: shorts_analysis/summary_stats_2021-2026_sample.json


In [58]:
# Final Summary Table
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

if results_df is not None:
    print(f"\n📈 SUMMARY TABLE:")
    print("-" * 80)

    n_long = int((results_df['connection_type'] == 'long').sum())
    n_short = int((results_df['connection_type'] == 'short').sum())
    n_none = int((results_df['connection_type'] == 'none').sum())

    summary_data = {
        'Category': [
            'Total Shorts',
            'Connected (any)',
            '  → to LONG video',
            '  → to SHORT video',
            'No connection',
            'Found via overlay',
            'Found via description',
        ],
        'Count': [
            len(results_df),
            len(results_df[results_df['status'] == 'connected']),
            n_long,
            n_short,
            n_none,
            int((results_df['detection_method'] == 'overlay_link').sum()),
            int((results_df['detection_method'] == 'description').sum()),
        ]
    }
    summary_table = pd.DataFrame(summary_data)
    summary_table['Percentage'] = (summary_table['Count'] / len(results_df) * 100).round(1).astype(str) + '%'
    print(summary_table.to_string(index=False))

    print(f"\n📁 All results saved to: ./shorts_analysis/")
    print(f"   - shorts_connections{suffix}.csv (all results)")
    if len(results_df[results_df['status'] == 'connected']) > 0:
        print(f"   - shorts_connection_mapping{suffix}.csv (recorded connections)")
    print(f"   - summary_stats{suffix}.json (statistics)")

    print(f"\n⏰ Date range analyzed: {START_YEAR} - {END_YEAR}")
    if USE_SAMPLE:
        print(f"⚠️  SAMPLE MODE: Processing {len(results_df)} shorts (stratified by channel)")
        print(f"   To analyze FULL dataset: Change USE_SAMPLE = False in first cell")


FINAL SUMMARY

📈 SUMMARY TABLE:
--------------------------------------------------------------------------------
             Category  Count Percentage
         Total Shorts     25     100.0%
      Connected (any)      3      12.0%
      → to LONG video      3      12.0%
     → to SHORT video      0       0.0%
        No connection     22      88.0%
    Found via overlay      0       0.0%
Found via description      0       0.0%

📁 All results saved to: ./shorts_analysis/
   - shorts_connections_2021-2026_sample.csv (all results)
   - shorts_connection_mapping_2021-2026_sample.csv (recorded connections)
   - summary_stats_2021-2026_sample.json (statistics)

⏰ Date range analyzed: 2021 - 2026
⚠️  SAMPLE MODE: Processing 25 shorts (stratified by channel)
   To analyze FULL dataset: Change USE_SAMPLE = False in first cell


In [59]:
import pandas as pd

# 1. Prepare the results for merging by renaming columns to be descriptive
merge_ready_results = results_df[["short_id", "connection_type", "connected_id", "detection_method"]].copy()
merge_ready_results.rename(columns={
    "short_id": "videoId",
    "connection_type": "extracted_connection_type",
    "connected_id": "extracted_connected_video_id",
    "detection_method": "extraction_method"
}, inplace=True)

# 2. Merge with the original metadata (yt_df) on videoId
# This aligns the analysis results with the original dataset rows
final_concatenated_df = pd.merge(
    yt_df,
    merge_ready_results,
    on="videoId",
    how="left"
)

# 3. Export to a single CSV for easy integration
output_path = "shorts_analysis/metadata_with_connections_final.csv"
final_concatenated_df.to_csv(output_path, index=False)

print(f"✅ Final joined dataset created: {output_path}")
print(f"New columns added: extracted_connection_type, extracted_connected_video_id, extraction_method")
display(final_concatenated_df[final_concatenated_df['extracted_connection_type'].notna()].head())

✅ Final joined dataset created: shorts_analysis/metadata_with_connections_final.csv
New columns added: extracted_connection_type, extracted_connected_video_id, extraction_method


,videoId,publishedAt,channelId,channelTitle,title,description,tags,categoryId,liveBroadcastContent,defaultLanguage,...,actualEndTime,scheduledStartTime,scheduledEndTime,concurrentViewers,activeLiveChatId,contentRating_ytRating,is_short,extracted_connection_type,extracted_connected_video_id,extraction_method
0,eNU5FcmEwTc,2026-02-27T14:16:42Z,UC4TFt-3rlYfhc_WtlWWY30Q,Camihawke,ChatGPT non ha aiutato...,NaN,NaN,22,none,it,...,NaN,NaN,NaN,NaN,NaN,NaN,True,none,NaN,NaN
79,gs7oJh6kJcY,2025-05-28T11:29:15Z,UCcH9hnbwzV5zrkuhZUeBgAw,Gli Autogol,PSG INTER - L'attesa,Juventini e milanisti si preparano così alla f...,NaN,17,none,it,...,NaN,NaN,NaN,NaN,NaN,NaN,True,none,NaN,NaN
989,VEBxxBohLHo,2023-11-30T11:41:42Z,UCkkcN04NIC0wwqNbcWlXNWQ,Canale di Venti,“Molliamo tutto e trasferiamoci in campagna” #...,NaN,NaN,24,none,it,...,NaN,NaN,NaN,NaN,NaN,NaN,True,long,WqRxEf7okik,overlay
2020,GRgp3BScD44,2023-03-28T11:15:47Z,UCF3eVR6iAiKYZHOXLj9Coyw,Cotto al Dente,Cestini di zucchine ricotta e gamberi. Ricetta...,NaN,NaN,17,none,it,...,NaN,NaN,NaN,NaN,NaN,NaN,True,none,NaN,NaN
2676,3mA4U7FNS80,2023-05-19T16:00:03Z,UCEwUio35yfUerEZdn1c9UWA,Raissa & Momo,"Comme te vene 'ncapa 'e di' ""I love you""? ❤️","Se ti è piaciuto il video, mi raccomando: iscr...","Vlog,Couple,Italian,Moroccan,Travel,Hd,4k,Gopr...",24,none,it,...,NaN,NaN,NaN,NaN,NaN,NaN,True,none,NaN,NaN


In [ ]:
# =========================================================================
# SAVE RESULTS BACK TO GCS  (Colab disk is ephemeral -> persist to the bucket)
# Mirrors the GCS load in the download cell; uploads the local outputs from
# ./shorts_analysis/ to gs://<GCP_BUCKET>/shorts_analysis/...
# =========================================================================
from google.cloud import storage
from pathlib import Path

GCS_OUTPUT_PREFIX = "shorts_analysis"  # folder inside the bucket

local_dir = Path('./shorts_analysis')
# Upload the joined metadata + every per-run artifact we produced above.
files_to_upload = sorted(
    p for p in local_dir.glob('*')
    if p.suffix in ('.csv', '.parquet', '.json') and not p.name.startswith('_')
)

if not files_to_upload:
    print("⚠️  Nothing to upload - run the processing/export cells first.")
else:
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCP_BUCKET)
    print(f"Uploading {len(files_to_upload)} file(s) to "
          f"gs://{GCP_BUCKET}/{GCS_OUTPUT_PREFIX}/ ...")
    for f in files_to_upload:
        blob = bucket.blob(f"{GCS_OUTPUT_PREFIX}/{f.name}")
        blob.upload_from_filename(str(f))
        print(f"  ✅ gs://{GCP_BUCKET}/{GCS_OUTPUT_PREFIX}/{f.name}")
    print("Done — results persisted to GCS.")